In [1]:
import os
import requests
import pandas as pd
from datetime import datetime

# =========================
# CONFIG
# =========================
OUTPUT_DIR = r"C:\xmas_project\outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

RAW_PATH   = os.path.join(OUTPUT_DIR, "tfl_raw_log.csv")
DAILY_PATH = os.path.join(OUTPUT_DIR, "tfl_daily_summary.csv")
SEV_PATH   = os.path.join(OUTPUT_DIR, "tfl_severity_rank.csv")

MODES = ["tube", "overground", "elizabeth-line", "dlr", "tram"]
URL = f"https://api.tfl.gov.uk/Line/Mode/{','.join(MODES)}/Status"

# Overground 2024 branded names + colours
OVERGROUND_MAP = {
    "lioness":     {"name": "Lioness",     "color": "#FFD200"},  # yellow
    "mildmay":     {"name": "Mildmay",     "color": "#0072CE"},  # blue
    "windrush":    {"name": "Windrush",    "color": "#E1251B"},  # red
    "weaver":      {"name": "Weaver",      "color": "#6F2DA8"},  # purple
    "suffragette": {"name": "Suffragette", "color": "#007A33"},  # green
    "liberty":     {"name": "Liberty",     "color": "#6E6E6E"}   # grey
}

# =========================
# FETCH
# =========================
def fetch_tfl_status():
    resp = requests.get(URL, timeout=30)
    resp.raise_for_status()
    return resp.json()

# =========================
# BUILD RAW LOG
# =========================
def build_raw_df(data):
    rows = []
    now_utc = datetime.utcnow()

    for line in data:
        mode = line.get("modeName")
        line_id = line.get("id")
        line_name = line.get("name")

        # Overground enrichment
        overground_line_name = None
        overground_color = None
        if mode == "overground":
            og = OVERGROUND_MAP.get(line_id)
            if og:
                overground_line_name = og["name"]
                overground_color = og["color"]

        for st in line.get("lineStatuses", []):
            rows.append({
                "timestamp_utc": now_utc,  # single snapshot time for the run
                "date": now_utc.date(),    # POWER BI relies on this

                "transport_mode": mode,
                "line_id": line_id,
                "line_name": line_name,

                # Overground branded fields
                "overground_line_name": overground_line_name,
                "overground_color": overground_color,

                "status": st.get("statusSeverityDescription"),
                "severity": st.get("statusSeverity"),
                "reason": st.get("reason")
            })

    df = pd.DataFrame(rows)

    # Hard typing for stability
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], errors="coerce")
    df["date"] = pd.to_datetime(df["date"], errors="coerce").dt.date
    df["severity"] = pd.to_numeric(df["severity"], errors="coerce")

    # Stable column order (helps Power BI)
    raw_cols = [
        "timestamp_utc", "date",
        "transport_mode", "line_id", "line_name",
        "overground_line_name", "overground_color",
        "status", "severity", "reason"
    ]
    df = df.reindex(columns=raw_cols)

    return df

# =========================
# BUILD DAILY SUMMARY
# =========================
def build_daily_summary(df):
    daily = (
        df.groupby(["date", "transport_mode", "line_name", "status"], dropna=False)
          .size()
          .reset_index(name="occurrences")
    )
    daily["occurrences"] = pd.to_numeric(daily["occurrences"], errors="coerce").fillna(0).astype(int)

    # Stable column order
    daily = daily.reindex(columns=["date", "transport_mode", "line_name", "status", "occurrences"])
    return daily

# =========================
# BUILD SEVERITY RANK
# =========================
def build_severity_rank(df):
    sev = (
        df.groupby(["transport_mode", "line_name"], dropna=False)
          .agg(
              avg_severity=("severity", "mean"),
              disruption_count=("severity", "count")
          )
          .reset_index()
    )

    # Hard typing
    sev["avg_severity"] = pd.to_numeric(sev["avg_severity"], errors="coerce")
    sev["disruption_count"] = pd.to_numeric(sev["disruption_count"], errors="coerce").fillna(0).astype(int)

    # Stable column order
    sev = sev.reindex(columns=["transport_mode", "line_name", "avg_severity", "disruption_count"])
    return sev

# =========================
# MAIN
# =========================
def main():
    print("Current working directory:", os.getcwd())
    print("Saving to:", OUTPUT_DIR)

    data = fetch_tfl_status()
    tfl_df = build_raw_df(data)

    # Sanity check
    print("\nMode counts:\n", tfl_df["transport_mode"].value_counts())

    tfl_daily_summary = build_daily_summary(tfl_df)
    tfl_severity_rank = build_severity_rank(tfl_df)

    # SAVE (overwrite same filenames Power BI already uses)
    tfl_df.to_csv(RAW_PATH, index=False, encoding="utf-8")
    tfl_daily_summary.to_csv(DAILY_PATH, index=False, encoding="utf-8")
    tfl_severity_rank.to_csv(SEV_PATH, index=False, encoding="utf-8")

    print("\n✅ Saved:")
    print(RAW_PATH, " | columns:", list(tfl_df.columns))
    print(DAILY_PATH, " | columns:", list(tfl_daily_summary.columns))
    print(SEV_PATH, " | columns:", list(tfl_severity_rank.columns))

if __name__ == "__main__":
    main()


Current working directory: C:\Users\Akash Basavaraju
Saving to: C:\xmas_project\outputs

Mode counts:
 transport_mode
tube              11
overground         6
dlr                1
elizabeth-line     1
tram               1
Name: count, dtype: int64

✅ Saved:
C:\xmas_project\outputs\tfl_raw_log.csv  | columns: ['timestamp_utc', 'date', 'transport_mode', 'line_id', 'line_name', 'overground_line_name', 'overground_color', 'status', 'severity', 'reason']
C:\xmas_project\outputs\tfl_daily_summary.csv  | columns: ['date', 'transport_mode', 'line_name', 'status', 'occurrences']
C:\xmas_project\outputs\tfl_severity_rank.csv  | columns: ['transport_mode', 'line_name', 'avg_severity', 'disruption_count']
